# 15. Queue & Circuit Breaker — resilience modules

`QueueModule` and `CircuitBreakerModule` are the two arcllm modules that
exist for one reason: **shaping how an agent talks to a fragile upstream
under load.**

- `QueueModule` answers *"how many calls am I allowed to have in flight, and
  what happens to caller #N+1?"* It enforces bounded concurrency with
  backpressure — at most `max_concurrent` calls hit the provider at once,
  the next `max_queued` wait their turn, and everyone else gets a clean
  `QueueFullError` instead of piling up unbounded work.
- `CircuitBreakerModule` answers *"the upstream is wedged — should I keep
  hammering it?"* It's a per-provider state machine (`CLOSED → OPEN →
  HALF_OPEN → CLOSED`) that opens after `failure_threshold` consecutive
  failures, fast-fails every call for `cooldown_seconds`, then probes
  recovery with a single half-open request.

Both are covered in passing in `04-agentic-loop` (as `load_model` kwargs)
and `07-module-system` (as positions in the canonical stack). **This
notebook is the reference: every public symbol, every state transition,
every error class, demonstrated end-to-end with a mock adapter.**

You will learn:

- How `max_concurrent` and `max_queued` interact — and why the timeout
  is *send-time only*, not wall-clock.
- When `QueueFullError` fires versus `QueueTimeoutError` — they look
  similar but mean very different things.
- Every transition in the circuit breaker state machine — including the
  one most demos skip (`HALF_OPEN → OPEN` on probe failure).
- Why the breaker is **per-provider** — and how that prevents a flaky
  Anthropic from taking down OpenAI in a multi-provider agent.
- How to compose `QueueModule` and `CircuitBreakerModule` together,
  which one catches what, and how to tune the knobs in production.


## 1. Setup

The standard arcllm walkthrough boilerplate. No API key needed — every
demo in this notebook uses a controllable mock adapter.

In [1]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

False

Pull in the public surface for both modules. Note the import paths:
`QueueModule`, `QueueFullError`, and `QueueTimeoutError` live on the
top-level `arcllm` package; `CircuitBreakerModule` and its companions
(`CircuitState`, `CircuitOpenError`) live under `arcllm.modules.circuit_breaker`.

In [2]:
import asyncio
import time

from arcllm import QueueFullError, QueueModule, QueueTimeoutError
from arcllm.exceptions import ArcLLMAPIError
from arcllm.modules.circuit_breaker import (
    CircuitBreakerModule,
    CircuitOpenError,
    CircuitState,
)
from arcllm.types import LLMProvider, LLMResponse, Message, Usage

print("QueueModule           :", QueueModule.__name__)
print("QueueFullError        :", QueueFullError.__name__)
print("QueueTimeoutError     :", QueueTimeoutError.__name__)
print("CircuitBreakerModule  :", CircuitBreakerModule.__name__)
print("CircuitState          :", list(CircuitState))
print("CircuitOpenError      :", CircuitOpenError.__name__)

QueueModule           : QueueModule
QueueFullError        : QueueFullError
QueueTimeoutError     : QueueTimeoutError
CircuitBreakerModule  : CircuitBreakerModule
CircuitState          : [<CircuitState.CLOSED: 'closed'>, <CircuitState.OPEN: 'open'>, <CircuitState.HALF_OPEN: 'half_open'>]
CircuitOpenError      : CircuitOpenError


## 2. Why resilience modules — flooding, cascading failure, flaky upstreams

LLM providers are flaky. They time out. They throttle. They have regional
outages. Without resilience modules, three failure modes ruin an agent:

| Failure | Symptom | Module that handles it |
|---|---|---|
| **Concurrency flood** | Caller fans out 100 LLM requests; provider 429s or returns 5xx; all 100 retry; storm. | `QueueModule` — bounded concurrency + reject excess |
| **Slow death** | Provider takes 30s per response; queue grows unbounded; memory spikes; OOM. | `QueueModule` — `max_queued` cap |
| **Cascading failure** | Provider is wedged returning 5xx; agent retries forever; tokens burn; SLO breached. | `CircuitBreakerModule` — open after N failures, fail fast |

**The split:**

- `QueueModule` is *load shaping*. It does not care whether the call
  succeeds — it only cares about how many are in flight.
- `CircuitBreakerModule` is *health gating*. It does not care about
  concurrency — it cares about consecutive failures.

In the canonical stack (see `07-module-system`), `QueueModule` sits near
the top so wait time is recorded by telemetry, and `CircuitBreakerModule`
sits below telemetry so a tripped breaker still emits a span:

```
Otel → Queue → Telemetry → CircuitBreaker → Audit → Security →
       Retry → Fallback → RateLimit → [Adapter]
```

We'll demonstrate the order and its consequences in section 9.


**The mock adapter.** Every demo below wraps a controllable mock
that implements the `LLMProvider` ABC. Two knobs: `latency_ms` (how long
each call takes) and `should_fail` (whether to raise). This lets us
reproduce throttling, slow upstreams, and outages deterministically.

In [3]:
class MockProvider(LLMProvider):
    """Controllable LLMProvider for resilience demos.

    Each call sleeps `latency_ms` then either returns a canned response
    or raises ArcLLMAPIError. Toggle `should_fail` to flip behavior
    mid-run; bump `call_count` to inspect how many times we hit upstream.
    """

    def __init__(
        self,
        name: str = "mock",
        latency_ms: float = 0.0,
        should_fail: bool = False,
    ) -> None:
        # LLMProvider.name / .model_name are abstract *properties* — a plain
        # `self.name = ...` instance assignment does not satisfy them (the
        # ABC check runs against the class, before __init__ ever executes).
        self._name = name
        self.latency_ms = latency_ms
        self.should_fail = should_fail
        self.call_count = 0

    @property
    def name(self) -> str:
        return self._name

    @property
    def model_name(self) -> str:
        return "mock-model-v1"

    async def invoke(self, messages, tools=None, **kwargs):
        self.call_count += 1
        if self.latency_ms > 0:
            await asyncio.sleep(self.latency_ms / 1000.0)
        if self.should_fail:
            raise ArcLLMAPIError(
                status_code=503,
                body="upstream unavailable",
                provider=self.name,
            )
        return LLMResponse(
            content=f"ok #{self.call_count}",
            usage=Usage(input_tokens=5, output_tokens=3, total_tokens=8),
            model=self.model_name,
            stop_reason="end_turn",
        )

    def validate_config(self) -> bool:
        return True


MSG = [Message(role="user", content="hi")]
print("MockProvider ready — fields:", ["name", "latency_ms", "should_fail", "call_count"])

MockProvider ready — fields: ['name', 'latency_ms', 'should_fail', 'call_count']


## 3. `QueueModule` basics — bounded concurrency

`QueueModule` wraps any `LLMProvider`. It exposes the same `invoke()`
API — the queue is transparent to callers. Internally:

- An `asyncio.BoundedSemaphore(max_concurrent)` gates execution.
- Excess callers wait FIFO until a slot frees up.
- A `_waiters` counter tracks queue depth so we can reject before
  acquiring the semaphore (cheaper than letting them queue and timing out).

Three config keys:

| Key | Default | Meaning |
|---|---|---|
| `max_concurrent` | `2` | Max in-flight calls to the inner provider |
| `max_queued` | `10` | Max waiters before backpressure kicks in |
| `call_timeout` | `180.0` | **Send-time** timeout (seconds) — see §5 |

Defaults are conservative because arcllm is federal-first — a runaway
agent shouldn't be able to flood a provider.

In [4]:
# Build a queue with max_concurrent=2 and watch parallelism cap there.
inner = MockProvider("queue-demo", latency_ms=100)
queue = QueueModule({"max_concurrent": 2, "max_queued": 10}, inner)

start = time.perf_counter()
results = await asyncio.gather(*[queue.invoke(MSG) for _ in range(4)])
elapsed_ms = (time.perf_counter() - start) * 1000

print(f"4 calls (~100ms each), max_concurrent=2 -> wall time ~{elapsed_ms:.0f}ms")
print(f"Inner adapter saw exactly {inner.call_count} calls (no rejections)")
print(f"Sample response: {results[0].content!r}")

4 calls (~100ms each), max_concurrent=2 -> wall time ~203ms
Inner adapter saw exactly 4 calls (no rejections)
Sample response: 'ok #2'


Two batches of two. With `max_concurrent=2` and 100ms latency, four
calls take ~200ms wall time, not 100ms (full parallelism would be 100ms,
serial would be 400ms). The semaphore enforces exactly the cap we asked
for — no more, no less.

**`queue_stats()` exposes runtime counters** for dashboards and
REST APIs. Useful in production when you want to chart queue health
without touching internals.

In [5]:
stats = queue.queue_stats()
for k, v in stats.items():
    print(f"  {k:>20}: {v}")

        max_concurrent: 2
            max_queued: 10
        call_timeout_s: 180.0
                active: 0
               waiting: 0
        total_enqueued: 4
       total_completed: 4
        total_rejected: 0
        total_timeouts: 0
           avg_wait_ms: 50.5


`active=0` and `waiting=0` because all four calls finished before
we read stats. `total_completed=4`, `avg_wait_ms` is small because the
semaphore became free quickly. The same dict is what `arcui` reads to
render the queue depth widget.

## 4. `QueueFullError` — when the queue is full

`max_concurrent` controls in-flight calls. `max_queued` controls how
many *additional* callers can be parked waiting. Together they bound
total memory pressure: at most `max_concurrent + max_queued` async
tasks own queue resources at any moment.

When `_waiters >= max_queued`, the next caller is rejected
**immediately** — no semaphore acquire, no provider hit. This is
backpressure: the agent learns "the queue is saturated" instantly
rather than after a 60-second timeout.

In [6]:
# max_concurrent=2, max_queued=2 → at most 4 calls own resources.
# Fire 10 concurrent → first 2 run, next 2 queue, last 6 reject.
inner = MockProvider("backpressure", latency_ms=200)
queue = QueueModule({"max_concurrent": 2, "max_queued": 2, "call_timeout": 5.0}, inner)


async def safe_invoke(i: int):
    try:
        await queue.invoke(MSG)
        return ("ok", i, None)
    except QueueFullError as exc:
        return ("rejected", i, exc)


results = await asyncio.gather(*[safe_invoke(i) for i in range(10)])

ok = [r for r in results if r[0] == "ok"]
rej = [r for r in results if r[0] == "rejected"]
print(f"Successful invocations  : {len(ok)}")
print(f"Rejected (QueueFullError): {len(rej)}")
print(f"Inner adapter calls     : {inner.call_count}  (only successful ones reach upstream)")

Queue backpressure: 2 waiters (max 2)


Queue backpressure: 2 waiters (max 2)


Queue backpressure: 2 waiters (max 2)


Queue backpressure: 2 waiters (max 2)


Queue backpressure: 2 waiters (max 2)


Queue backpressure: 2 waiters (max 2)


Successful invocations  : 4
Rejected (QueueFullError): 6
Inner adapter calls     : 4  (only successful ones reach upstream)


Four pass through, six are rejected — exactly what the math
predicts (`max_concurrent + max_queued = 4`). Critically, the inner
adapter only saw **4 calls**: the 6 rejected callers never touched the
upstream. That's the value of backpressure — your provider's 429 budget
isn't being burned by traffic the agent already knows it can't serve.

**`QueueFullError` carries structured fields** so the caller can
log, alert, or apply admission control.

In [7]:
# Demonstrate the structured fields on QueueFullError
example_err = rej[0][2]
print(f"Type            : {type(example_err).__name__}")
print(f"current_waiters : {example_err.current_waiters}")
print(f"max_queued      : {example_err.max_queued}")
print(f"str(err)        : {example_err}")

Type            : QueueFullError
current_waiters : 2
max_queued      : 2
str(err)        : Queue full: 2 calls waiting (max 2)


**Operational note:** if you see `QueueFullError` in production,
you're not necessarily in trouble — you're enforcing your SLA. The
question is whether you wanted to enforce it *here* or further upstream
(e.g. at an ingress rate limiter). When `QueueFullError` rate exceeds a
few percent, raise `max_queued` or scale out agents.

## 5. `QueueTimeoutError` — send-time timeout (not queue-wait)

The most surprising thing about `QueueModule.call_timeout` is that it
is **send-time only**: the clock starts *after* the semaphore is
acquired. It measures provider response time, not the time the caller
spent waiting in the queue.

This is intentional. If a slow provider is keeping the queue busy,
penalising waiters with a queue-wait timeout punishes the wrong actor —
the queue is doing its job. We want to know: "once a call escaped the
queue, how long did the provider take?"

This separation also means a long but legitimate streaming response
won't be killed because the queue happened to be deep at submission
time.

In [8]:
# Inner takes 500ms, but call_timeout is 100ms — should raise.
inner = MockProvider("slow-upstream", latency_ms=500)
queue = QueueModule({"max_concurrent": 2, "call_timeout": 0.1, "max_queued": 5}, inner)

t0 = time.perf_counter()
try:
    await queue.invoke(MSG)
    raise AssertionError("expected QueueTimeoutError")
except QueueTimeoutError as exc:
    elapsed = (time.perf_counter() - t0) * 1000
    print(f"Caught QueueTimeoutError after {elapsed:.0f}ms (timeout=100ms)")
    print(f"  err.timeout = {exc.timeout!r}")
    print(f"  str(err)    = {exc}")

Queue send-time timeout after 0.1s


Caught QueueTimeoutError after 101ms (timeout=100ms)
  err.timeout = 0.1
  str(err)    = LLM call timed out after 0.1s (send-time)


**Compare and contrast.** `QueueFullError` fires when too many
callers are waiting. `QueueTimeoutError` fires when one in-flight call
takes too long. Different problems, different alerts.

In [9]:
# Side-by-side comparison: both raised by the same module, different causes
print("""
QueueFullError:    triggered before the semaphore is acquired.
                   Cause: too many waiters. Caller arrives, queue is full,
                   immediate rejection. Inner provider never sees the call.

QueueTimeoutError: triggered after the semaphore is acquired.
                   Cause: provider too slow. Caller is in flight, but
                   asyncio.wait_for cancels after `call_timeout`.
                   Inner provider DID see the call.
""")


QueueFullError:    triggered before the semaphore is acquired.
                   Cause: too many waiters. Caller arrives, queue is full,
                   immediate rejection. Inner provider never sees the call.

QueueTimeoutError: triggered after the semaphore is acquired.
                   Cause: provider too slow. Caller is in flight, but
                   asyncio.wait_for cancels after `call_timeout`.
                   Inner provider DID see the call.



**Stats let you tell which is happening.**

In [10]:
# Build queue stats showing both kinds of failure
inner = MockProvider("slow", latency_ms=300)
q = QueueModule({"max_concurrent": 1, "max_queued": 1, "call_timeout": 0.05}, inner)


async def attempt():
    try:
        await q.invoke(MSG)
    except (QueueFullError, QueueTimeoutError):
        pass


await asyncio.gather(*[attempt() for _ in range(6)])

s = q.queue_stats()
print(f"total_enqueued : {s['total_enqueued']:>3}  (calls accepted into queue)")
print(f"total_completed: {s['total_completed']:>3}  (calls that returned a response)")
print(f"total_rejected : {s['total_rejected']:>3}  (QueueFullError — never reached provider)")
print(f"total_timeouts : {s['total_timeouts']:>3}  (QueueTimeoutError — provider too slow)")

Queue backpressure: 1 waiters (max 1)


Queue backpressure: 1 waiters (max 1)


Queue backpressure: 1 waiters (max 1)


Queue backpressure: 1 waiters (max 1)


Queue send-time timeout after 0.1s


Queue send-time timeout after 0.1s


total_enqueued :   2  (calls accepted into queue)
total_completed:   0  (calls that returned a response)
total_rejected :   4  (QueueFullError — never reached provider)
total_timeouts :   2  (QueueTimeoutError — provider too slow)


`total_rejected` and `total_timeouts` are independent counters —
the right alert dashboard charts them separately.

## 6. `CircuitBreakerModule` basics — the state machine

A circuit breaker has three states. Picture a literal household breaker:

- **`CLOSED`** — current flows. Calls pass through normally. The breaker
  counts consecutive failures.
- **`OPEN`** — breaker is tripped. Every call raises `CircuitOpenError`
  *without* hitting the provider. Stays open for `cooldown_seconds`.
- **`HALF_OPEN`** — cooldown elapsed. Allow `half_open_max_calls`
  through as a probe. If they succeed → `CLOSED`. If any fail → back to
  `OPEN`.

Configuration:

| Key | Default | Meaning |
|---|---|---|
| `failure_threshold` | `5` | Consecutive failures before opening |
| `cooldown_seconds` | `30.0` | How long to stay open before probing |
| `half_open_max_calls` | `1` | Probe budget in half-open |
| `on_state_change` | `None` | Optional `(old, new, info) -> None` callback |
| `on_event` | `None` | Optional callback receiving `TraceRecord(circuit_change)` |

Defaults are deliberately gentle: you usually want stricter thresholds
once you've measured baseline error rates. We'll tune in §10.

In [11]:
# Healthy provider, breaker closed → calls pass through.
inner = MockProvider("healthy")
breaker = CircuitBreakerModule({"failure_threshold": 3, "cooldown_seconds": 0.5}, inner)

resp = await breaker.invoke(MSG)
print(f"Response       : {resp.content!r}")
print(f"Inner saw      : {inner.call_count} call(s)")
state = breaker.get_state()
for k, v in state.items():
    print(f"  {k:>22}: {v}")

Response       : 'ok #1'
Inner saw      : 1 call(s)
                provider: healthy
                   model: mock-model-v1
                   state: closed
    consecutive_failures: 0
       last_failure_time: None
       failure_threshold: 3
        cooldown_seconds: 0.5
     half_open_max_calls: 1


`get_state()` is the queryable read-only snapshot — what the
arcui dashboard polls and what `arc breaker status` would print.
`state == 'closed'`, `consecutive_failures == 0`, healthy.

**`CircuitState` is a `StrEnum`** with three members. You can
compare against the string literal or the enum — same value.

In [12]:
# CircuitState — the public enum
print("Members:")
for member in CircuitState:
    print(f"  {member.name:<10} = {member.value!r}")

print()
print("StrEnum semantics — both comparisons work:")
print(f"  CircuitState.CLOSED == 'closed'  → {CircuitState.CLOSED == 'closed'}")
print(f"  state['state'] == CircuitState.CLOSED → {state['state'] == CircuitState.CLOSED}")

Members:
  CLOSED     = 'closed'
  OPEN       = 'open'
  HALF_OPEN  = 'half_open'

StrEnum semantics — both comparisons work:
  CircuitState.CLOSED == 'closed'  → True
  state['state'] == CircuitState.CLOSED → True


## 7. State transitions — driving the full machine

Now the main event. We'll drive the breaker through every transition
and assert the state at each step.

**Transitions to demonstrate:**

1. `CLOSED → OPEN`  — N consecutive failures trip the breaker.
2. *(while open)* — calls fast-fail with `CircuitOpenError`.
3. `OPEN → HALF_OPEN` — cooldown elapses, next call is allowed as a probe.
4. `HALF_OPEN → CLOSED` — probe succeeds, breaker resets.
5. `HALF_OPEN → OPEN` — probe fails, breaker re-opens (a fresh cooldown).

We'll capture the transitions with the `on_state_change` callback so
we can replay the sequence.

In [13]:
# Capture every transition into a list for inspection.
transitions: list[tuple[str, str]] = []


def record(old: str, new: str, info: dict) -> None:
    transitions.append((old, new))
    print(f"  state change: {old:>9} → {new:<9}  (failures={info['consecutive_failures']})")


inner = MockProvider("flaky")
breaker = CircuitBreakerModule(
    {
        "failure_threshold": 3,
        "cooldown_seconds": 0.3,
        "half_open_max_calls": 1,
        "on_state_change": record,
    },
    inner,
)
print("Initial state:", breaker.get_state()["state"])

Initial state: closed


### 7a. CLOSED → OPEN (failure_threshold reached)

Force the inner to fail. After the third consecutive failure, the
breaker opens.

In [14]:
inner.should_fail = True

for i in range(1, 4):
    try:
        await breaker.invoke(MSG)
    except ArcLLMAPIError:
        print(f"call {i}: provider raised → state now {breaker.get_state()['state']!r}")

assert breaker.get_state()["state"] == CircuitState.OPEN

call 1: provider raised → state now 'closed'
call 2: provider raised → state now 'closed'
  state change:    closed → open       (failures=3)
call 3: provider raised → state now 'open'


3 failures, exactly 1 transition: `closed → open`. The breaker
counted the consecutive failures and tripped.

### 7b. OPEN — fast-fail without hitting upstream

While open, the breaker rejects every call **immediately** with
`CircuitOpenError`. The inner provider does not get hit — that's the
whole point.

In [15]:
# Reset the call counter so we can prove the inner isn't touched
inner.call_count = 0

try:
    await breaker.invoke(MSG)
except CircuitOpenError as exc:
    print(f"Caught {type(exc).__name__}")
    print(f"  provider          : {exc.provider!r}")
    print(f"  cooldown_remaining: {exc.cooldown_remaining:.3f}s")
    print(f"  Inner call_count  : {inner.call_count}  (zero — provider unreached)")

Caught CircuitOpenError
  provider          : 'flaky'
  cooldown_remaining: 0.297s
  Inner call_count  : 0  (zero — provider unreached)


### 7c. OPEN → HALF_OPEN → CLOSED (probe succeeds)

Wait for cooldown to elapse, flip the inner back to healthy, fire one
call. The breaker transitions to `HALF_OPEN` to probe, the call
succeeds, and the breaker closes.

In [16]:
# Wait past the cooldown
await asyncio.sleep(0.35)
inner.should_fail = False  # provider has recovered

resp = await breaker.invoke(MSG)
print(f"Probe response     : {resp.content!r}")
print(f"Final state        : {breaker.get_state()['state']!r}")
print(f"Consecutive failures reset: {breaker.get_state()['consecutive_failures']}")

  state change:      open → half_open  (failures=3)
  state change: half_open → closed     (failures=0)
Probe response     : 'ok #1'
Final state        : 'closed'
Consecutive failures reset: 0


Two transitions in one call: `open → half_open` (cooldown elapsed,
probe budget granted) and `half_open → closed` (probe succeeded). The
failure counter resets to 0 — back to a clean slate.

### 7d. HALF_OPEN → OPEN (probe fails)

The most interesting case: what if the probe fails? The breaker
re-opens immediately and a fresh cooldown begins. We don't need
N more failures — *one* probe failure is enough.

In [17]:
# Trip the breaker again, then make the probe fail.
inner.should_fail = True
transitions.clear()

# 3 failures → OPEN
for _ in range(3):
    try:
        await breaker.invoke(MSG)
    except ArcLLMAPIError:
        pass
print(f"After failures: state={breaker.get_state()['state']!r}")

# wait through cooldown — next call will be a probe
await asyncio.sleep(0.35)

# probe still fails because should_fail is True
try:
    await breaker.invoke(MSG)
except ArcLLMAPIError:
    print(f"Probe failed   : state={breaker.get_state()['state']!r}")

print()
print("Transitions in this round:")
for old, new in transitions:
    print(f"  {old:>10} → {new}")

  state change:    closed → open       (failures=3)
After failures: state='open'


  state change:      open → half_open  (failures=3)
  state change: half_open → open       (failures=4)
Probe failed   : state='open'

Transitions in this round:
      closed → open
        open → half_open
   half_open → open


You can see the full path: `closed → open → half_open → open`.
The breaker doesn't need three more failures — one failed probe is
enough to re-open. This prevents flapping into HALF_OPEN repeatedly
when a provider is genuinely down.

### `on_event` — TraceRecord stream for audit

The same transitions are also emitted as `TraceRecord(event_type=
'circuit_change')` via the `on_event` callback. This is what the
audit module and the arcui dashboard subscribe to — same data,
structured form.

In [18]:
from arcllm.trace_store import TraceRecord

records: list[TraceRecord] = []
inner2 = MockProvider("audited", latency_ms=0)
inner2.should_fail = True
breaker2 = CircuitBreakerModule(
    {"failure_threshold": 2, "cooldown_seconds": 30.0, "on_event": records.append},
    inner2,
)

for _ in range(2):
    try:
        await breaker2.invoke(MSG)
    except ArcLLMAPIError:
        pass

assert len(records) == 1
rec = records[0]
print(f"event_type : {rec.event_type}")
print(f"provider   : {rec.provider}")
print(f"old_state  : {rec.event_data['old_state']}")
print(f"new_state  : {rec.event_data['new_state']}")
print(f"failures   : {rec.event_data['consecutive_failures']}")

event_type : circuit_change
provider   : audited
old_state  : closed
new_state  : open
failures   : 2


These records flow through the arcllm trace store (notebook
`14-trace-store`) and become part of the hash-chained audit trail.

## 8. Per-provider isolation — one breaker per upstream

A single `CircuitBreakerModule` instance wraps **one** provider. If you
have a multi-provider agent (Anthropic + OpenAI + a self-hosted vLLM),
you instantiate one breaker per provider. They share no state.

This is the right design. A flaky Anthropic should not cause OpenAI
calls to fast-fail — that would be cascading failure, the exact thing
the breaker exists to prevent.

`load_model(provider="anthropic", circuit_breaker={...})` builds one
breaker scoped to that provider; another `load_model("openai", ...)`
call builds an independent one. Below we simulate both by hand.

In [19]:
# Build two independent breakers, each wrapping its own MockProvider.
anthropic_inner = MockProvider("anthropic")
openai_inner = MockProvider("openai")

anthropic_breaker = CircuitBreakerModule(
    {"failure_threshold": 2, "cooldown_seconds": 5.0}, anthropic_inner
)
openai_breaker = CircuitBreakerModule(
    {"failure_threshold": 2, "cooldown_seconds": 5.0}, openai_inner
)

# Anthropic goes down. OpenAI stays healthy.
anthropic_inner.should_fail = True
openai_inner.should_fail = False

# Trip the anthropic breaker
for _ in range(2):
    try:
        await anthropic_breaker.invoke(MSG)
    except ArcLLMAPIError:
        pass

print(f"anthropic state: {anthropic_breaker.get_state()['state']}")
print(f"openai    state: {openai_breaker.get_state()['state']}")

anthropic state: open
openai    state: closed


Anthropic is open. OpenAI is still closed. Now prove that OpenAI
calls succeed *and* that Anthropic calls fast-fail — the two breakers
are completely independent.

In [20]:
# OpenAI calls work normally
resp = await openai_breaker.invoke(MSG)
print(f"OpenAI call worked: {resp.content!r}")

# Anthropic calls fast-fail (don't even hit the provider)
anthropic_inner.call_count = 0
try:
    await anthropic_breaker.invoke(MSG)
except CircuitOpenError as exc:
    print(f"Anthropic call rejected: {exc}")
    print(f"  Anthropic provider hit {anthropic_inner.call_count} times (zero — fast-fail)")

OpenAI call worked: 'ok #1'
Anthropic call rejected: Circuit open for 'anthropic': 5.0s remaining in cooldown
  Anthropic provider hit 0 times (zero — fast-fail)


**Why this matters in production.** A multi-provider agent with
fallback (`fallback=["openai", "anthropic"]`) relies on this isolation.
If breakers were shared, a single bad provider would brick the whole
agent. With per-provider breakers, the fallback module observes
`CircuitOpenError` from anthropic and routes to openai — cascading
failure becomes graceful degradation.

## 9. Composing both — Queue *outside* CircuitBreaker

The canonical arcllm stack puts `QueueModule` near the top and
`CircuitBreakerModule` lower:

```
Otel → Queue → Telemetry → CircuitBreaker → Audit → Security →
       Retry → Fallback → RateLimit → [Adapter]
```

In wrapping order this means **`Queue` wraps `CircuitBreaker`** which
wraps the adapter. A request flows top-to-bottom; the response flows
bottom-to-top.

**What each catches:**

- `QueueModule` outermost: rejects before counting against the breaker.
  A `QueueFullError` does not move the failure counter — the call never
  reached the provider.
- `CircuitBreakerModule` further in: tracks **provider** health. A
  `CircuitOpenError` raised by the breaker *does* count as queue
  activity (it consumed a slot).

This means: if your provider is wedged, the breaker opens after N
failures, and subsequent calls fast-fail at the breaker — but they
still spend a slot in the queue (briefly). That's intentional: the
queue's job is to bound concurrency, regardless of what the inner
modules do with the call.

In [21]:
# Build the canonical sandwich manually: Queue -> CircuitBreaker -> Adapter
inner = MockProvider("composed")
breaker = CircuitBreakerModule({"failure_threshold": 3, "cooldown_seconds": 30.0}, inner)
queue = QueueModule({"max_concurrent": 2, "max_queued": 4, "call_timeout": 5.0}, breaker)

# Healthy path: everything passes through both layers.
inner.should_fail = False
resp = await queue.invoke(MSG)
print(f"Healthy call : {resp.content!r}")
print(
    f"Queue stats  : completed={queue.queue_stats()['total_completed']}, "
    f"rejected={queue.queue_stats()['total_rejected']}"
)
print(f"Breaker state: {breaker.get_state()['state']!r}")

Healthy call : 'ok #1'
Queue stats  : completed=1, rejected=0
Breaker state: 'closed'


Now flip the provider to fail and watch how the two errors stack.
The first failures count against the breaker; once the breaker opens,
subsequent calls raise `CircuitOpenError` from the breaker but still
pass through the queue's bookkeeping.

In [22]:
# Drive 5 calls into a failing inner. Watch the failure mix.
inner.should_fail = True
inner.call_count = 0

errors: dict[str, int] = {"api_error": 0, "circuit_open": 0, "queue_full": 0}


async def attempt():
    try:
        await queue.invoke(MSG)
    except CircuitOpenError:
        errors["circuit_open"] += 1
    except QueueFullError:
        errors["queue_full"] += 1
    except ArcLLMAPIError:
        errors["api_error"] += 1


await asyncio.gather(*[attempt() for _ in range(5)])

print(f"ArcLLMAPIError    (provider failed)         : {errors['api_error']}")
print(f"CircuitOpenError  (breaker fast-failed)     : {errors['circuit_open']}")
print(f"QueueFullError    (queue saturated)         : {errors['queue_full']}")
print()
print(f"Inner provider was hit  : {inner.call_count} times")
print(f"Breaker state           : {breaker.get_state()['state']}")

ArcLLMAPIError    (provider failed)         : 3
CircuitOpenError  (breaker fast-failed)     : 2
QueueFullError    (queue saturated)         : 0

Inner provider was hit  : 3 times
Breaker state           : open


You can see the breaker doing its job: only `failure_threshold`
calls actually reached the provider. After that, `CircuitOpenError`
fast-fails subsequent calls. The queue counted them all (because the
queue's bookkeeping happens at the outer layer), but the upstream was
spared.

**Key consequence:** `CircuitOpenError` is the breaker's signal,
`QueueFullError` is the queue's signal, `ArcLLMAPIError` is the
provider's. Three different problems, three different alerts.

**Does composition order matter?** Yes — the canonical stack puts
Queue outside CircuitBreaker. Reversing the order means a tripped
breaker doesn't hold a queue slot, which sounds nice but breaks the
"Queue == load shaping" invariant: telemetry's queue depth chart would
no longer reflect actual concurrency. The default exists for a reason.

## 10. Operational guidance — tuning and alerting

Defaults are conservative. Once you've measured baseline behavior,
tune to your traffic. Rules of thumb:

### `QueueModule`

| Knob | Tuning approach |
|---|---|
| `max_concurrent` | Start at 2 per agent. Raise until you see provider-side throttling, then back off 20%. Multiply by N agents at the gateway, not per agent. |
| `max_queued` | At least `max_concurrent`. Raise if you see `QueueFullError` rate exceed 1% during normal load. Lower in latency-sensitive paths (fail fast). |
| `call_timeout` | Default 180s matches the adapter HTTP timeout. Lower for chat UIs (~30s); raise for long-running structured generation. |

**Alert on:**

- `total_rejected / total_enqueued > 0.05` for 5 min — queue is undersized
  or upstream is too slow.
- `total_timeouts / total_completed > 0.01` — provider is degrading.
- `avg_wait_ms > <SLO budget>` — concurrency is undersized.

### `CircuitBreakerModule`

| Knob | Tuning approach |
|---|---|
| `failure_threshold` | Default 5 is for stable providers. For flaky cloud APIs, lower to 3. For self-hosted models you trust, raise to 10. |
| `cooldown_seconds` | Default 30s. Match the provider's incident MTTR. Too short = thrashing; too long = unnecessary downtime. |
| `half_open_max_calls` | Default 1. Increase to 2-3 only if your provider has caching behavior that rewards a brief warm-up. |

**Alert on:**

- Any `circuit_change` event with `new_state == "open"` — page someone.
- Repeated `half_open → open` transitions — the cooldown is too short
  for the provider's actual recovery time.
- `consecutive_failures` climbing without opening — failure_threshold
  is too high for your traffic.

The two modules complement each other: the queue contains the load
exposure, the breaker contains the failure exposure. Together they
turn a flooded, flaky upstream into a bounded, observable problem.

In [23]:
# Production-style configuration — start point, then measure and tune.
prod_inner = MockProvider("prod-anthropic")
prod_breaker = CircuitBreakerModule(
    {
        "failure_threshold": 3,  # tighter than default 5
        "cooldown_seconds": 30.0,
        "half_open_max_calls": 1,
    },
    prod_inner,
)
prod_queue = QueueModule(
    {
        "max_concurrent": 4,  # 2x default — measured against provider RPS
        "max_queued": 8,  # 2x max_concurrent → headroom for bursts
        "call_timeout": 60.0,  # 60s for chat-style traffic
    },
    prod_breaker,
)

# Smoke test: prove the stack still works under realistic call volume.
prod_inner.should_fail = False
results = await asyncio.gather(*[prod_queue.invoke(MSG) for _ in range(8)])

print(f"Calls completed : {len(results)}")
qs = prod_queue.queue_stats()
print(f"  total_enqueued: {qs['total_enqueued']}")
print(f"  total_completed: {qs['total_completed']}")
print(f"  avg_wait_ms   : {qs['avg_wait_ms']}")
print(f"Breaker state   : {prod_breaker.get_state()['state']!r}")

Calls completed : 8
  total_enqueued: 8
  total_completed: 8
  avg_wait_ms   : 0.0
Breaker state   : 'closed'


**The `load_model` shortcut.** In a real agent, you wire this
through `arcllm.load_model(circuit_breaker={...}, queue={...})` rather
than constructing modules by hand. The kwarg pattern is covered in
`04-agentic-loop`. The configuration dicts are exactly the ones we used
above — same keys, same defaults.

In [24]:
# Reference: equivalent load_model invocation (commented — needs a real provider).
example = """
from arcllm import load_model

provider = load_model(
    "anthropic",
    "claude-sonnet-4-6",
    queue={
        "max_concurrent": 4,
        "max_queued": 8,
        "call_timeout": 60.0,
    },
    circuit_breaker={
        "failure_threshold": 3,
        "cooldown_seconds": 30.0,
        "half_open_max_calls": 1,
    },
    # other modules...
)
"""
print(example.strip())

from arcllm import load_model

provider = load_model(
    "anthropic",
    "claude-sonnet-4-6",
    queue={
        "max_concurrent": 4,
        "max_queued": 8,
        "call_timeout": 60.0,
    },
    circuit_breaker={
        "failure_threshold": 3,
        "cooldown_seconds": 30.0,
        "half_open_max_calls": 1,
    },
    # other modules...
)


## 11. Summary

`QueueModule` and `CircuitBreakerModule` are the two arcllm resilience
modules. They solve different problems and compose into a layered
defence against load and failure.

**What you saw:**

- `QueueModule` enforces bounded concurrency via
  `asyncio.BoundedSemaphore(max_concurrent)`. Excess callers wait FIFO
  up to `max_queued`; beyond that, `QueueFullError` rejects them
  immediately without touching upstream.
- `call_timeout` is *send-time only* — the clock starts after the
  semaphore is acquired. `QueueTimeoutError` means a slow provider, not
  a deep queue.
- `queue_stats()` exposes runtime counters (`total_enqueued`,
  `total_completed`, `total_rejected`, `total_timeouts`, `avg_wait_ms`)
  for dashboards and REST APIs.
- `CircuitBreakerModule` is a `CLOSED → OPEN → HALF_OPEN` state machine.
  After `failure_threshold` consecutive failures it opens for
  `cooldown_seconds`; the next call is a half-open probe;
  success → `CLOSED`, failure → `OPEN` again.
- Every transition is observable via `on_state_change` (callback) and
  `on_event` (TraceRecord stream). The same data flows into the audit
  trail and the arcui dashboard.
- Per-provider isolation: one `CircuitBreakerModule` per upstream. A
  flaky Anthropic does not affect OpenAI calls.
- Canonical composition: **Queue outside CircuitBreaker**. Queue does
  load shaping, breaker does health gating; reversing the order breaks
  the telemetry contract.

**Public API covered:**

- `arcllm.QueueModule` — bounded concurrency wrapper.
- `arcllm.QueueFullError` — backpressure rejection (`current_waiters`,
  `max_queued`).
- `arcllm.QueueTimeoutError` — send-time timeout (`timeout`).
- `QueueModule.queue_stats()` — runtime counter snapshot.
- `arcllm.modules.circuit_breaker.CircuitBreakerModule` — state machine.
- `arcllm.modules.circuit_breaker.CircuitState` — the `CLOSED`,
  `OPEN`, `HALF_OPEN` enum (`StrEnum`).
- `arcllm.modules.circuit_breaker.CircuitOpenError` — fast-fail
  exception (`provider`, `cooldown_remaining`).
- `CircuitBreakerModule.get_state()` — queryable state dict.
- `on_state_change(old, new, info)` and
  `on_event(TraceRecord)` callbacks.

**Next:**

- `04-agentic-loop` — wiring queue and circuit breaker through
  `load_model(queue=..., circuit_breaker=...)`.
- `07-module-system` — the canonical module stack and why the order
  matters.
- `09-telemetry-module` — how queue wait time and breaker state changes
  appear in the telemetry stream.
- `14-trace-store` — where `circuit_change` TraceRecords land for
  long-term audit.
- `arcrun/01-core-react` — using a queue-and-breaker-wrapped provider
  inside a real agent loop.